![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [39]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

In [40]:
with open("data/car_reviews.csv", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        if i in (0, 1, 2, 3):
            print(i, repr(line))
        if i > 3:
            break

0 '\ufeffReview;Class\n'
1 'I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use. Camping, road trips, etc. We dont have any children so I store most of the seats in my warehouse. I wanted the passenger van for the rear air conditioning. We drove our van from Florida to California for a Cross Country trip in 2014. We averaged about 18 mpg. We drove thru a lot of rain and It was a very comfortable and stable vehicle. The V8 Nissan Titan engine is a 500k mile engine. It has been tested many times by delivery and trucking companies. This is why Nissan gives you a 5 year or 100k mile bumper to bumper warranty. Many people are scared about driving this van because of its size. But with front and rear sonar sensors, large mirrors and the back up camera. It is easy to drive. The front and rear sensors also monitor the front and rear sides of the bumpers making it easier to park close to objects. Our Nissan NV is a Tow Monster. It pulls our 

In [41]:
df = pd.read_csv("data/car_reviews.csv", sep=";", encoding="utf-8-sig")
df.head()

,Review,Class
0,I am very satisfied with my 2014 Nissan NV SL....,POSITIVE
1,The car is fine. It's a bit loud and not very ...,NEGATIVE
2,"My first foreign car. Love it, I would buy ano...",POSITIVE
3,I've come across numerous reviews praising the...,NEGATIVE
4,I've been dreaming of owning an SUV for quite ...,POSITIVE


In [42]:
# Start your code here!
from transformers import pipeline
import evaluate

classifier = pipeline(task="sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

predicted_labels = classifier(df["Review"].tolist())

predictions = [1 if pred["label"] == "POSITIVE" else 0 for pred in predicted_labels]
references = [1 if label == "POSITIVE" else 0 for label in df["Class"]]

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

accuracy_result = accuracy_metric.compute(predictions=predictions, references=references)["accuracy"]
f1_result = f1_metric.compute(predictions=predictions, references=references)["f1"]

print(accuracy_result)
print(f1_result)


Device set to use cpu


0.8
0.8571428571428571


In [43]:
import re

first_review = df["Review"].iloc[0]

sentences = re.split(r'(?<=[.!?]) +', first_review)
two_sentences = " ".join(sentences[:2])
print(two_sentences)

I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use.


In [44]:
with open("data/reference_translations.txt", encoding="utf-8") as f:
    references = [line.strip() for line in f if line.strip()]
print(references)
print(len(references))


['Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.', 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.']
2


In [45]:
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")

translation_result = translator(two_sentences)
translated_review = translation_result[0]["translation_text"]
print(translated_review)

Device set to use cpu


Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal.


In [46]:
bleu_metric = evaluate.load("bleu")

bleu_score = bleu_metric.compute(
    predictions=[translated_review],
    references=[references]   
)
print(bleu_score)

{'bleu': 0.7794483794144497, 'precisions': [0.9090909090909091, 0.8571428571428571, 0.75, 0.631578947368421], 'brevity_penalty': 1.0, 'length_ratio': 1.0476190476190477, 'translation_length': 22, 'reference_length': 21}


In [47]:
qa_model = pipeline("question-answering", model="deepset/minilm-uncased-squad2")

question = "What did he like about the brand?"
context = df["Review"].iloc[1]

result = qa_model(question=question, context=context)
answer = result["answer"]
print(answer)

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


ride quality, reliability


In [48]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

last_review = df["Review"].iloc[-1]

summary_result = summarizer(last_review, max_length=55, min_length=50, do_sample=False)
summarized_text = summary_result[0]["summary_text"]
print(summarized_text)

Device set to use cpu


 Nissan Rogue provides the desired SUV experience without burdening me with an exorbitant payment . Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more . The engine delivers strong performance, and
